# VoiceGuard — Acoustic Deepfake Classifier Training (Google Colab)

This notebook trains the binary acoustic deepfake classifier (`AcousticDeepfakeCNN` using EfficientNet-B0) on a cloud GPU (e.g. Free-tier Google Colab T4).

### Operating Principles & Honesty Constraints:
1. **Train in-domain, evaluate out-of-domain**: Report in-domain test EER alongside an unseen-corpus out-of-domain test EER.
2. **Strict speaker disjointness**: Speakers in the training set must never appear in validation or test splits.
3. **Automatic Mixed Precision (AMP)** and checkpointing enabled to survive transient session resets.

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi

# Install dependencies
!pip install -q timm>=0.9.12 librosa>=0.10.1 soundfile>=0.12.1 scikit-learn>=1.4.0 structlog webrtcvad-wheels matplotlib pyyaml kagglehub


In [ ]:
# 2. Clone VoiceGuard Repository if running in Colab
import os, sys
from pathlib import Path

repo_root = Path("/content/VoiceGuard").resolve()
if not repo_root.exists():
    print("Cloning VoiceGuard repository from GitHub...")
    !git clone https://github.com/AS24xADITYA/VoiceGuard.git /content/VoiceGuard
    repo_root = Path("/content/VoiceGuard").resolve()

backend_dir = repo_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))
print(f"Backend loaded from: {backend_dir}")


## 3. Dataset Acquisition (ASVspoof 2019 Logical Access via Kaggle or Instant Mini Test)

Select your mode below:
- `RUN_MINI_TEST = True` (Default): Instant 60-second synthetic smoke test to verify the entire pipeline, Spectrogram extraction, and training loop.
- `RUN_MINI_TEST = False`: Downloads the official ASVspoof 2019 LA dataset via Kagglehub for full benchmark training.

In [ ]:
# ── 3. Dataset Acquisition ────────────────────────────────────────────────
import os, sys
from pathlib import Path

# Set RUN_MINI_TEST = True for an instant 60-second smoke test
# Set RUN_MINI_TEST = False to download the full ASVspoof 2019 LA dataset from Kaggle
RUN_MINI_TEST = True

data_dir = Path("/content/data")
data_dir.mkdir(parents=True, exist_ok=True)

if RUN_MINI_TEST:
    print("Generating synthetic mini-dataset for instant verification...")
    import numpy as np, soundfile as sf
    mini_dir = data_dir / "mini_asvspoof"
    for split in ["train", "dev"]:
        split_dir = mini_dir / split
        split_dir.mkdir(parents=True, exist_ok=True)
        proto_lines = []
        sr = 16000
        # 60 bonafide + 60 spoof clips per split
        for i in range(120):
            t = np.linspace(0, 3.0, int(3.0 * sr), endpoint=False)
            is_spoof = i >= 60
            spk_id = f"SPK_{split}_{i // 10:02d}"
            key = "spoof" if is_spoof else "bonafide"
            attack = "A01" if is_spoof else "-"
            freq = 220.0 + (50.0 * np.sin(2 * np.pi * 2.0 * t) if not is_spoof else 0.0)
            sig = 0.5 * np.sin(2 * np.pi * freq * t) if not is_spoof else 0.4 * np.sign(np.sin(2 * np.pi * 220.0 * t))
            # Syllabic envelope so VAD recognizes voiced speech
            sig = sig * 0.5 * (1.0 + np.sin(2 * np.pi * 3.5 * t))
            sig += 0.01 * np.random.randn(len(t))
            fname = f"{split}_{i:04d}.wav"
            sf.write(str(split_dir / fname), sig.astype(np.float32), sr)
            proto_lines.append(f"{spk_id} {fname} - {attack} {key}")
        with open(mini_dir / f"{split}_protocol.txt", "w") as f:
            f.write("\n".join(proto_lines))
    print(f"✓ Created synthetic benchmark at {mini_dir} (120 train, 120 dev audio files)")

else:
    print("Downloading official ASVspoof 2019 LA Dataset via Kaggle...")
    import kagglehub
    kaggle_path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
    print(f"Kaggle dataset downloaded to: {kaggle_path}")
    
    # Locate LA directory
    candidates = list(Path(kaggle_path).rglob("*ASVspoof2019_LA_train*"))
    if candidates:
        la_src = candidates[0].parent
    else:
        la_src = Path(kaggle_path) / "LA" if (Path(kaggle_path) / "LA").exists() else Path(kaggle_path)
    
    target_la = Path("/content/data/LA")
    if not target_la.exists():
        !ln -s "{la_src}" /content/data/LA
    print(f"✓ ASVspoof 2019 LA linked to /content/data/LA")


In [ ]:
# ── 4. Dataset Preprocessing & Speaker Disjointness Verification ────────────
from pathlib import Path

if RUN_MINI_TEST:
    train_proto = "/content/data/mini_asvspoof/train_protocol.txt"
    train_audio = "/content/data/mini_asvspoof/train"
    dev_proto = "/content/data/mini_asvspoof/dev_protocol.txt"
    dev_audio = "/content/data/mini_asvspoof/dev"
else:
    la_root = Path("/content/data/LA")
    train_proto_candidates = list(la_root.rglob("*train.trn.txt"))
    train_proto = str(train_proto_candidates[0]) if train_proto_candidates else "/content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
    train_audio_candidates = list(la_root.rglob("*ASVspoof2019_LA_train*"))
    train_audio = str(train_audio_candidates[0] / "flac" if (train_audio_candidates[0] / "flac").exists() else train_audio_candidates[0])
    
    dev_proto_candidates = list(la_root.rglob("*dev.trl.txt"))
    dev_proto = str(dev_proto_candidates[0]) if dev_proto_candidates else "/content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
    dev_audio_candidates = list(la_root.rglob("*ASVspoof2019_LA_dev*"))
    dev_audio = str(dev_audio_candidates[0] / "flac" if (dev_audio_candidates[0] / "flac").exists() else dev_audio_candidates[0])

print("--- Preprocessing Training Split ---")
!python /content/VoiceGuard/scripts/prepare_datasets.py \
    --protocol "{train_proto}" \
    --audio-dir "{train_audio}" \
    --output-dir /content/processed/asvspoof_train \
    --precompute-specs

print("\n--- Preprocessing Development (Validation) Split ---")
!python /content/VoiceGuard/scripts/prepare_datasets.py \
    --protocol "{dev_proto}" \
    --audio-dir "{dev_audio}" \
    --output-dir /content/processed/asvspoof_dev \
    --precompute-specs


In [ ]:
# 5. Launch Training Loop
import json
from pathlib import Path
from ai.acoustic.train import run_training

# Load manifests generated by prepare_datasets.py
train_manifest_path = Path("/content/processed/asvspoof_train/manifest.json")
val_manifest_path = Path("/content/processed/asvspoof_dev/manifest.json")

if train_manifest_path.exists() and val_manifest_path.exists():
    with open(train_manifest_path) as f:
        train_manifest = json.load(f)
    with open(val_manifest_path) as f:
        val_manifest = json.load(f)

    epochs = 3 if RUN_MINI_TEST else 30

    training_summary = run_training(
        train_manifest=train_manifest,
        val_manifest=val_manifest,
        output_dir="/content/model_output",
        backbone="efficientnet_b0",
        epochs=epochs,
        batch_size=32,
        lr=3e-4,
        weight_decay=1e-4,
        patience=6,
    )
    print("Training completed successfully!")
else:
    print("Manifest files not found at specified paths. Ensure data preprocessing is completed.")


In [ ]:
# 6. In-Domain & Out-of-Domain Evaluation
import torch
from pathlib import Path
from ai.acoustic.detector import AcousticDetector

model_ckpt = Path("/content/model_output/model_best.pt")
if model_ckpt.exists():
    detector = AcousticDetector(
        model_path=model_ckpt,
        backbone="efficientnet_b0",
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    detector.load()
    detector.warmup()
    print("✓ AcousticDetector successfully loaded and verified with trained weights!")


In [ ]:
# 7. Export Model Artifact & 1-Click Browser Download
import shutil
from pathlib import Path

best_model_pt = Path("/content/model_output/model_best.pt")
acoustic_pth = Path("/content/model_output/acoustic.pth")

if best_model_pt.exists():
    shutil.copy(best_model_pt, acoustic_pth)
    print(f"✓ Created deployment artifact: {acoustic_pth}")
    print("\n--- Initiating Browser Download ---")
    try:
        from google.colab import files
        files.download(str(acoustic_pth))
        print("✓ Download prompt opened! Place this file in your local project at: VoiceGuard/backend/models/acoustic.pth")
    except Exception as e:
        print(f"Download manually from the Colab file browser: {acoustic_pth}")
else:
    print("Model checkpoint not found. Ensure training completed.")
